# 05 — Mentorship Decay Index (behavioral, from existing data)  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Set PROJECT_ID → **Runtime → Run all**. Reads only your own `frame_events_*` — cheap, no API, no GH Archive.

Measures mentorship/onboarding **behaviorally**, at ecosystem scale, without needing comment text:

- **Newcomer** = a contributor's *first* opened PR or issue in a repo (within 30 days of their first activity there).
- **Newcomer Response Rate (NRR)** = fraction of newcomers whose first contribution received a response from *someone else* (non-bot) within 90 days.
- **Mentorship Decay** = NRR falling over time — the community engaging less with the people trying to join.

This is the fifth isolation signal and the one most directly about "society." Comment *text* is reserved for the later 500-interaction annotation, which validates that these responses are genuinely helpful/mentoring.


## Config — edit PROJECT_ID

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
RELIABLE_END = "202506"; KNOWN_GAP_MONTHS = ["202110"]
BQ_DATASET = "seis"; BQ_LOCATION = "US"; MAX_SCAN_GB = 500
DRIVE_BASE = "/content/drive/MyDrive/ICSE_SEIS"
assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded.")


## Setup — auth, BigQuery, Drive, helpers

In [ ]:
import os, json, hashlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from google.colab import auth, drive
from google.cloud import bigquery
auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
for sub in ["", "/figures", "/tables", "/checkpoints"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
CKPT_DIR=f"{DRIVE_BASE}/checkpoints"
def _a(p,t):
    tmp=p+".tmp"; open(tmp,"w").write(t); os.replace(tmp,p)
def load_json(n,d):
    p=f"{CKPT_DIR}/{n}"; return json.load(open(p)) if os.path.exists(p) else d
def save_json(n,o): _a(f"{CKPT_DIR}/{n}", json.dumps(o,default=str))
_jr=load_json("bq_jobs.json",{})
def _sjr(): save_json("bq_jobs.json",_jr)
def run_query(sql, destination=None, write="WRITE_TRUNCATE", to_dataframe=False, label=None, job_key=None, cluster=None):
    tag=f"[{label}] " if label else ""
    h=hashlib.md5(sql.encode()).hexdigest()[:8]
    if job_key and _jr.get(job_key,{}).get("hash")==h:
        try:
            prev=client.get_job(_jr[job_key]["job_id"], location=BQ_LOCATION)
            if prev.state=="DONE" and prev.error_result is None:
                print(f"{tag}reuse job"); r=prev.result(); return r.to_dataframe() if to_dataframe else r
        except Exception as e: print(f"{tag}(reattach failed: {e})")
    gb=client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True)).total_bytes_processed/1e9
    print(f"{tag}scan ~{gb:,.1f} GB (~${gb/1000*6.25:,.2f})")
    cfg=bigquery.QueryJobConfig(maximum_bytes_billed=int(max(MAX_SCAN_GB,gb*1.2)*1e9))
    if destination: cfg.destination=f"{PROJECT_ID}.{BQ_DATASET}.{destination}"; cfg.write_disposition=write
    if cluster: cfg.clustering_fields=cluster
    job=client.query(sql, job_config=cfg)
    if job_key: _jr[job_key]={"job_id":job.job_id,"hash":h}; _sjr()
    r=job.result(); print(f"{tag}done -> {destination or 'df'}")
    return r.to_dataframe() if to_dataframe else r
P=f"{PROJECT_ID}.{BQ_DATASET}"; FE=f"`{PROJECT_ID}.{BQ_DATASET}.frame_events_*`"
print("Ready.", P)


## 1. Build the newcomer table
For every contributor, find their first opened PR/issue in each repo (restricted to genuine newcomers — first thread within 30 days of first activity), then count responses from others (non-bot) on that thread within 90 days. One row per newcomer.

In [ ]:
mdi_sql = f"""
WITH fe AS (SELECT * FROM {FE}),
acts AS (
  SELECT repo_id, actor_login, MIN(created_at) AS first_seen
  FROM fe
  WHERE NOT is_bot AND actor_login IS NOT NULL
    AND type IN ('PushEvent','PullRequestEvent','IssuesEvent','IssueCommentEvent',
                 'PullRequestReviewCommentEvent','CommitCommentEvent')
  GROUP BY 1,2),
opens AS (
  SELECT repo_id, actor_login AS newcomer, number, MIN(created_at) AS open_time
  FROM fe
  WHERE type IN ('PullRequestEvent','IssuesEvent') AND action='opened'
        AND NOT is_bot AND number IS NOT NULL
  GROUP BY 1,2,3),
first_thread AS (
  SELECT repo_id, newcomer, number, open_time FROM (
    SELECT o.*, ROW_NUMBER() OVER (PARTITION BY repo_id, newcomer ORDER BY open_time) rn
    FROM opens o) WHERE rn=1),
newc AS (
  SELECT ft.* FROM first_thread ft JOIN acts a
    ON ft.repo_id=a.repo_id AND ft.newcomer=a.actor_login
  WHERE ft.open_time <= TIMESTAMP_ADD(a.first_seen, INTERVAL 30 DAY))
SELECT n.repo_id, n.newcomer, n.number, n.open_time,
       FORMAT_TIMESTAMP('%Y%m', n.open_time) AS ym,
       COUNTIF(e.actor_login != n.newcomer AND NOT e.is_bot) AS responses
FROM newc n
LEFT JOIN fe e
  ON e.repo_id=n.repo_id AND e.number=n.number
 AND e.type IN ('IssueCommentEvent','PullRequestReviewCommentEvent')
 AND e.created_at BETWEEN n.open_time AND TIMESTAMP_ADD(n.open_time, INTERVAL 90 DAY)
GROUP BY 1,2,3,4,5
"""
run_query(mdi_sql, destination="mdi_newcomers", label="MDI-newcomers",
          job_key="mdi_newcomers", cluster=["repo_id"])
print("mdi_newcomers built.")


## 2. Repo-month MDI table (for the panel)

In [ ]:
repo_mdi_sql = f"""
SELECT repo_id, ym,
  COUNT(*) AS newcomers,
  AVG(IF(responses>0,1,0)) AS nrr,
  AVG(responses) AS avg_responses
FROM `{P}.mdi_newcomers`
GROUP BY repo_id, ym
"""
run_query(repo_mdi_sql, destination="metric_mdi", label="metric_mdi",
          job_key="metric_mdi", cluster=["repo_id"])
print("metric_mdi built.")


## 3. Ecosystem NRR time series (newcomer-weighted) + plot
Newcomer-weighted (not repo-averaged) so big and small repos count by their actual newcomer inflow. Declining NRR = mentorship decay.

In [ ]:
ts_sql = f"""
SELECT ym,
  COUNT(*) AS newcomers,
  AVG(IF(responses>0,1,0)) AS nrr,
  AVG(responses) AS avg_responses
FROM `{P}.mdi_newcomers`
WHERE ym BETWEEN '201801' AND '{RELIABLE_END}'
  AND ym NOT IN UNNEST({KNOWN_GAP_MONTHS})
GROUP BY ym ORDER BY ym
"""
mdi = run_query(ts_sql, to_dataframe=True, label="MDI-ts", job_key="mdi_ts")
mdi["date"] = pd.to_datetime(mdi["ym"]+"01", format="%Y%m%d")
mdi.to_csv(f"{DRIVE_BASE}/tables/mdi_timeseries.csv", index=False)

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(mdi.date, mdi.nrr); ax[0].axvline(pd.Timestamp("2022-11-01"), ls="--", color="crimson")
ax[0].set_title("Newcomer Response Rate (NRR) — mentorship"); ax[0].grid(alpha=.3)
ax[1].plot(mdi.date, mdi.newcomers); ax[1].set_title("Newcomers per month (inflow)"); ax[1].grid(alpha=.3)
fig.suptitle("Mentorship / onboarding engagement, 2018–2025", y=1.03); fig.tight_layout()
fig.savefig(f"{DRIVE_BASE}/figures/mdi_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved mdi_trend.png"); mdi.tail()


## Done ✅
Built `mdi_newcomers` (newcomer-level), `metric_mdi` (repo-month), and the NRR time series (+ figure).

Read NRR as: does a first-time contributor's opening PR/issue get a human response? A **declining NRR** means the community is engaging less with newcomers — mentorship decay, a core SEIS concern.

Next: **`06_networks.ipynb`** — contributor interaction-network structure over time (density, clustering, centralization), then annotation + the statistical models.

In [ ]:
mdi["year"] = mdi["ym"].str[:4]
print(mdi.groupby("year")[["nrr","avg_responses","newcomers"]].mean().round(4))